# M1 — Extensão 2: CVRP completo (relaxando a rota-estrela)

**Treinamento de Otimização Aplicada — Genoa para Gradus**

Este notebook contém **apenas a parte CVRP** do M1 — extraída de `m1_frota.ipynb` para foco didático e rapidez no Colab.

## Da rota-estrela ao CVRP

No caso simples do M1 usamos a aproximação **rota-estrela** (ida-volta direta ao bar mais distante). Aqui relaxamos isso e o solver decide a ordem real de visita.

Comparamos **3 abordagens**:

| Abordagem | Tempo | Quando usar |
|---|---|---|
| OR-Tools `RoutingModel` | ~5 s (GLS) | Heurística rápida, até centenas de clientes |
| Gurobi MTZ | ~500 ms | Formulação MILP polinomial, dezenas de clientes |
| Gurobi DFJ + lazy constraints | ~1 s | Padrão industrial, escala melhor |

## Setup

In [ ]:
!pip install -q ortools gurobipy pandas

In [ ]:
import math, time
import pandas as pd

# Coordenadas aproximadas (lat, lon) — CD + 8 bares em SP
COORDS = {
    'CD':            (-23.567, -46.685),
    'Centro':        (-23.553, -46.635),
    'Pinheiros':     (-23.565, -46.685),
    'Vila Madalena': (-23.555, -46.692),
    'Moema':         (-23.605, -46.665),
    'Tatuapé':       (-23.539, -46.572),
    'Lapa':          (-23.521, -46.706),
    'Itaim':         (-23.583, -46.671),
    'Brooklin':      (-23.612, -46.690),
}
NODES = list(COORDS.keys())
N = len(NODES)
DEPOT = 0
CUSTOMERS = list(range(1, N))

# Demanda (caixas/semana)
DEM = [0, 55, 30, 50, 45, 60, 35, 40, 25]

# Parâmetros do caminhão (mesmos do caso simples)
Q = 100             # capacidade
RENT = 2000         # R$/semana aluguel
CKM = 4             # R$/km combustível
MAX_TRUCKS = 6      # limite superior de veículos

# Matriz de distâncias 9×9 (km)
def km(a, b): return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111
D = {(i, j): km(COORDS[NODES[i]], COORDS[NODES[j]]) for i in range(N) for j in range(N)}

# Tabela de demanda + distância ao CD (para conferir)
print(f"Demanda total: {sum(DEM)} caixas/semana")
print(f"Distância ao CD (km):")
for i, n in enumerate(NODES):
    if i == 0: continue
    print(f"  {n:<14}: {D[0, i]:>5.1f} km  ·  {DEM[i]} cx")

## 1) OR-Tools `RoutingModel` — solver dedicado a VRP

Vantagens:
- API pensada pra roteamento (não é "forçar" como em PL/MIP)
- Heurísticas robustas (PATH_CHEAPEST_ARC, SAVINGS) + busca local (Guided Local Search)
- Escala bem (até centenas de clientes)

Desvantagem:
- Não dá garantia de otimalidade — é heurística

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# Distâncias em décimos de km (OR-Tools quer inteiros)
Dint = [[int(round(D[i, j] * 10)) for j in range(N)] for i in range(N)]

# Setup do roteador
mgr = pywrapcp.RoutingIndexManager(N, MAX_TRUCKS, DEPOT)
rt = pywrapcp.RoutingModel(mgr)

# Callback de distância (custo de cada arco)
def dist_cb(from_idx, to_idx):
    return Dint[mgr.IndexToNode(from_idx)][mgr.IndexToNode(to_idx)]
transit = rt.RegisterTransitCallback(dist_cb)
rt.SetArcCostEvaluatorOfAllVehicles(transit)

# Callback de demanda (com dimensão de capacidade)
def demand_cb(from_idx):
    return DEM[mgr.IndexToNode(from_idx)]
demand_idx = rt.RegisterUnaryTransitCallback(demand_cb)
rt.AddDimensionWithVehicleCapacity(demand_idx, 0, [Q]*MAX_TRUCKS, True, 'Capacity')

# Custo fixo por veículo usado (R$ 2.000 = 5.000 décimos de unidade de FO)
for v in range(MAX_TRUCKS):
    rt.SetFixedCostOfVehicle(5000, v)   # encoraja usar menos caminhões

# Search params
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.seconds = 5

t0 = time.time()
sol = rt.SolveWithParameters(params)
t_or = time.time() - t0

# Extrair rotas
total_km_or = 0
n_veic_or = 0
rotas_or = []
for v in range(MAX_TRUCKS):
    idx = rt.Start(v)
    if rt.IsEnd(sol.Value(rt.NextVar(idx))): continue   # rota vazia
    n_veic_or += 1
    rota = []
    while not rt.IsEnd(idx):
        rota.append(NODES[mgr.IndexToNode(idx)])
        nxt = sol.Value(rt.NextVar(idx))
        total_km_or += Dint[mgr.IndexToNode(idx)][mgr.IndexToNode(nxt)] / 10
        idx = nxt
    rota.append('CD')
    rotas_or.append(' → '.join(rota))

custo_or = 2000 * n_veic_or + 4 * total_km_or
print(f'OR-Tools RoutingModel — solução encontrada em {t_or:.1f}s')
print(f'  {n_veic_or} caminhões · {total_km_or:.1f} km · custo R$ {custo_or:,.2f}')
for r in rotas_or:
    print(f'  - {r}')

## 2) Gurobi MTZ — formulação MILP polinomial

Variáveis:
- $x_{ijk} \in \{0,1\}$: caminhão $k$ usa o arco $i \to j$
- $y_k \in \{0,1\}$: caminhão $k$ é alugado
- $u_{ik} \ge 1$: ordem de visita do bar $i$ pelo caminhão $k$ (MTZ)

Restrição-chave de subtour elimination:

$$u_{ik} - u_{jk} + (n-1) \cdot x_{ijk} \le n - 2 \quad \forall i, j \ne \text{CD}, k$$

Quando $x_{ijk} = 1$, força $u_{jk} \ge u_{ik} + 1$ — sequência monotônica.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

m = gp.Model('cvrp_mtz')
m.Params.OutputFlag = 0
m.Params.TimeLimit = 30

# Variáveis
x = m.addVars(range(N), range(N), range(MAX_TRUCKS), vtype=GRB.BINARY, name='x')
y = m.addVars(range(MAX_TRUCKS), vtype=GRB.BINARY, name='y')
u = m.addVars(CUSTOMERS, range(MAX_TRUCKS), lb=1, ub=N-1, name='u')

# Cobertura: cada cliente visitado exatamente 1 vez por algum caminhão
m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(MAX_TRUCKS)) == 1
              for j in CUSTOMERS), 'visit')

# Conservação de fluxo: o que entra sai
m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
              gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
              for j in range(N) for k in range(MAX_TRUCKS)), 'flow')

# Início: caminhão sai do depot no máximo 1 vez (= y[k])
m.addConstrs((gp.quicksum(x[DEPOT,j,k] for j in CUSTOMERS) == y[k]
              for k in range(MAX_TRUCKS)), 'start')

# Capacidade
m.addConstrs((gp.quicksum(DEM[j] * x[i,j,k] for i in range(N) for j in CUSTOMERS if i!=j) <= Q * y[k]
              for k in range(MAX_TRUCKS)), 'cap')

# Sem self-loops
m.addConstrs((x[i,i,k] == 0 for i in range(N) for k in range(MAX_TRUCKS)), 'noself')

# MTZ subtour elimination
for k in range(MAX_TRUCKS):
    for i in CUSTOMERS:
        for j in CUSTOMERS:
            if i != j:
                m.addConstr(u[i,k] - u[j,k] + (N-1)*x[i,j,k] <= N-2)

# Objetivo: aluguel + combustível
m.setObjective(
    RENT * gp.quicksum(y[k] for k in range(MAX_TRUCKS))
    + CKM * gp.quicksum(D[i,j] * x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(MAX_TRUCKS)),
    GRB.MINIMIZE
)

t0 = time.time()
m.optimize()
t_mtz = time.time() - t0

# Extrair rotas
rotas_mtz = []
for k in range(MAX_TRUCKS):
    if y[k].X < 0.5: continue
    cur = DEPOT
    rota = [NODES[cur]]
    while True:
        prox = next((j for j in range(N) if j != cur and x[cur,j,k].X > 0.5), None)
        if prox is None or prox == DEPOT:
            rota.append('CD'); break
        rota.append(NODES[prox]); cur = prox
    rotas_mtz.append(' → '.join(rota))

custo_mtz = m.ObjVal
print(f'Gurobi MTZ — ótimo provado em {t_mtz*1000:.0f} ms')
print(f'  custo R$ {custo_mtz:,.2f}')
for r in rotas_mtz:
    print(f'  - {r}')

## 3) Gurobi DFJ + lazy constraints — o killer feature

**Ideia:** em vez de adicionar TODAS as restrições de subtour upfront (exponencial), partir SEM elas e adicionar só as que aparecerem violadas via *callback*.

**Como o callback funciona:**
1. Solver acha solução candidata (com possível subtour)
2. Callback inspeciona — há subset $S$ desconectado do depot?
3. Se sim: adiciona a constraint $\sum_{i,j \in S} x_{ij} \le |S| - 1$
4. Solver continua a busca com a nova restrição
5. Repete até nenhum subtour aparecer

Para 50–100+ clientes, isso é tipicamente o que viabiliza o CVRP exato. Solvers open-source não têm callbacks robustos.

In [ ]:
def subtour_elim_callback(model, where):
    """Detecta subtours na solução corrente e adiciona DFJ constraints."""
    if where != GRB.Callback.MIPSOL:
        return
    for k in range(MAX_TRUCKS):
        # Recupera valores binários da solução corrente
        arcs_idx = [(i, j) for i in range(N) for j in range(N) if i != j]
        vals = model.cbGetSolution([model._x[i, j, k] for i, j in arcs_idx])
        arcs = [(i, j) for (i, j), v in zip(arcs_idx, vals) if v > 0.5]
        if not arcs: continue

        # BFS a partir do depot — quem está alcançável faz parte da rota legítima
        visitados = {DEPOT}
        stack = [DEPOT]
        while stack:
            n = stack.pop()
            for (i, j) in arcs:
                if i == n and j not in visitados:
                    visitados.add(j); stack.append(j)

        # Nós que aparecem em arcs mas não foram alcançados = subtour
        todos = {i for (i, _) in arcs} | {j for (_, j) in arcs}
        subtour = todos - visitados
        if subtour:
            S = list(subtour)
            model.cbLazy(
                gp.quicksum(model._x[i, j, k] for i in S for j in S if i != j) <= len(S) - 1
            )

m2 = gp.Model('cvrp_dfj_lazy')
m2.Params.OutputFlag = 0
m2.Params.TimeLimit = 30
m2.Params.LazyConstraints = 1   # OBRIGATÓRIO para cbLazy funcionar

x2 = m2.addVars(range(N), range(N), range(MAX_TRUCKS), vtype=GRB.BINARY)
y2 = m2.addVars(range(MAX_TRUCKS), vtype=GRB.BINARY)
m2._x = x2   # disponível dentro do callback

# Mesmas restrições do MTZ, MAS SEM MTZ
m2.addConstrs((gp.quicksum(x2[i,j,k] for i in range(N) if i!=j for k in range(MAX_TRUCKS)) == 1
               for j in CUSTOMERS))
m2.addConstrs((gp.quicksum(x2[i,j,k] for i in range(N) if i!=j) ==
               gp.quicksum(x2[j,i,k] for i in range(N) if i!=j)
               for j in range(N) for k in range(MAX_TRUCKS)))
m2.addConstrs((gp.quicksum(x2[DEPOT,j,k] for j in CUSTOMERS) == y2[k] for k in range(MAX_TRUCKS)))
m2.addConstrs((gp.quicksum(DEM[j] * x2[i,j,k] for i in range(N) for j in CUSTOMERS if i!=j) <= Q * y2[k]
               for k in range(MAX_TRUCKS)))
m2.addConstrs((x2[i,i,k] == 0 for i in range(N) for k in range(MAX_TRUCKS)))

m2.setObjective(
    RENT * gp.quicksum(y2[k] for k in range(MAX_TRUCKS))
    + CKM * gp.quicksum(D[i,j] * x2[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(MAX_TRUCKS)),
    GRB.MINIMIZE
)

t0 = time.time()
m2.optimize(subtour_elim_callback)
t_dfj = time.time() - t0

rotas_dfj = []
for k in range(MAX_TRUCKS):
    if y2[k].X < 0.5: continue
    cur = DEPOT
    rota = [NODES[cur]]
    while True:
        prox = next((j for j in range(N) if j != cur and x2[cur,j,k].X > 0.5), None)
        if prox is None or prox == DEPOT:
            rota.append('CD'); break
        rota.append(NODES[prox]); cur = prox
    rotas_dfj.append(' → '.join(rota))

custo_dfj = m2.ObjVal
print(f'Gurobi DFJ + lazy callbacks — ótimo em {t_dfj*1000:.0f} ms')
print(f'  custo R$ {custo_dfj:,.2f}')
for r in rotas_dfj:
    print(f'  - {r}')

## Comparação final das 3 abordagens

In [ ]:
comp = pd.DataFrame([
    {'método': 'OR-Tools RoutingModel',         'custo': custo_or,  'tempo (ms)': int(t_or * 1000)},
    {'método': 'Gurobi MTZ (formulação polinomial)', 'custo': custo_mtz, 'tempo (ms)': int(t_mtz * 1000)},
    {'método': 'Gurobi DFJ + lazy constraints',  'custo': custo_dfj, 'tempo (ms)': int(t_dfj * 1000)},
])
comp['custo'] = comp['custo'].apply(lambda v: f'R$ {v:,.2f}')
comp

## Comparação com a rota-estrela do caso simples

Da solução do caso simples (notebook `m1_frota.ipynb`):
- **Rota-estrela:** R$ 8.201,60 (custo estimado, otimista)
- **CVRP (qualquer abordagem):** R$ 8.250,64 (custo real)

**Diferença: R$ 49** — a rota-estrela subestimava o custo de combustível porque ignorava as distâncias entre bares dentro de cada cluster.

Em problemas pequenos (8 bares), a diferença é pequena. Em redes reais (50+ bares), a rota-estrela pode subestimar em **10–30 %** — aí o CVRP exato vale o esforço.

## Lições deste notebook

1. **OR-Tools RoutingModel** é o caminho mais rápido para CVRP até 100s de clientes. Heurística boa, código simples.
2. **Gurobi MTZ** dá garantia de otimalidade, mas a formulação polinomial fica frouxa em escala — degrada após ~50 clientes.
3. **Gurobi DFJ + lazy constraints** combina otimalidade com escalabilidade. É o padrão da indústria para CVRP exato.
4. **Lazy constraints é recurso de solver comercial** — CBC e GLPK não suportam callbacks de forma robusta. É um dos motivos clássicos para investir em licença Gurobi/CPLEX em projetos de roteamento.

👉 Para um caso **industrial** (40 bares, onde MTZ no Gurobi free *falha* por exceder o limite de 2 000 variáveis), veja `m1_frota_industrial.ipynb`.